<a href="https://colab.research.google.com/github/Samkas/TextMiningWS25/blob/main/Lesson%201/nltk_gensim_WS25_janout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load Data

For our exercises we will use two different datasets:
- The AG News subset that features english news articles of 4 different categories
    - https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset
- The 10kGNAD dataset that features german news articles of 9 different categories
    - https://www.kaggle.com/datasets/mexwell/10kgnad

In [ ]:
import csv
import pandas as pd
from typing import List, Set, Tuple

# english data
classes_en = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
train_en = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/AGNews/train.csv",
                       names = ["Label", "Title", "Article"],
                       encoding = "utf-8")
test_en = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/AGNews/test.csv",
                      names = ["Label", "Title", "Article"],
                      encoding = "utf-8")

# german data
train_de = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/10kGNAD/train.csv",
                       sep = ";", names = ["Label", "Article"],
                       quotechar = "\'", quoting = csv.QUOTE_MINIMAL, encoding = "utf-8")
test_de = pd.read_csv("https://raw.githubusercontent.com/Samkas/TextMiningWS25/refs/heads/main/data/10kGNAD/test.csv",
                       sep = ";", names = ["Label", "Article"],
                       quotechar = "\'", quoting = csv.QUOTE_MINIMAL, encoding = "utf-8")

We can iterate of the dataframe cols to construct a custom list of documents to work on

In [ ]:
labels_en = [classes_en[int(row["Label"])] for i, row in train_en.iterrows()]
articles_en = [row["Article"] for i, row in train_en.iterrows()]
labels_de = [row["Label"] for i, row in train_de.iterrows()]
articles_de = [row["Article"] for i, row in train_de.iterrows()]

In [ ]:
articles_en[:5]


["Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
 'Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.',
 'Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.',
 'AFP - Tearaway world oil prices, toppling records and straining wallets, present a new economic menace barely three months before the US presidential elections.']

## NLTK Overview

https://www.nltk.org/ is a leading Python platform for working with human language data. It provides:

- Interfaces to 50+ corpora and lexical resources (e.g., WordNet)
- Text processing libraries for:
  - Tokenization
  - Stemming
  - Tagging
  - Parsing
  - Semantic reasoning
- Wrappers for industrial-strength NLP libraries

NLTK is widely used for research, education, and prototyping in natural language processing.

---

### What We Will Do

We will preprocess English and German articles using NLTK by applying:

1. Tokenization – Splitting text into words or sentences.
2. Stemming – Reducing words to their root form.
3. Stopword Removal – Removing common words that carry little meaning.

After each step, we’ll inspect how the text changes, so you can see the transformation clearly.

This mirrors the steps we previously saw in SpaCy, where much of this was automated, but here we’ll perform them manually to understand each process in detail and see how NLTK can be used.

In [ ]:
# import required packages
import nltk
from nltk.corpus import stopwords as nltkStopwords
from nltk.stem.snowball import SnowballStemmer

### NLTK tokenizes documents which are any string variables

In [ ]:
# download nltk resources
nltk.download("punkt") #  tokenizer
nltk.download("punkt_tab") # required for tokenization
nltk.download("stopwords") # stopword list


# tokenize the document
# contrary to spacey, nltk requires to tokenize each document separately
# we can either tokenize per word or sentence, each has its own advantages and disadvantages
articles_en_tokenized = [nltk.word_tokenize(doc) for doc in articles_en]
articles_de_tokenized = [nltk.word_tokenize(doc) for doc in articles_de]

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\p42011\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\p42011\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\p42011\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
articles_en_tokenized[0]

['Reuters',
 '-',
 'Short-sellers',
 ',',
 'Wall',
 'Street',
 "'s",
 'dwindling\\band',
 'of',
 'ultra-cynics',
 ',',
 'are',
 'seeing',
 'green',
 'again',
 '.']

### Stemming can be done with NLTK's Snowball Stemming

[https://www.nltk.org/api/nltk.stem.snowball.html](https://www.nltk.org/api/nltk.stem.snowball.html)

We will write a function for this to loop over each article of our list

In [ ]:
# we need to define a function to apply the snowballstemmer to our document
def stem(tokenized_document: list[str], language: str | None = None) -> List[str]:
    # Initialize the Snowball stemmer for the specified language
    # `ignore_stopwords=False` ensures stopwords are not excluded from stemming
    stemmer = SnowballStemmer(language, ignore_stopwords=False)

    # Apply stemming to each word in the tokenized document
    # Assumes `tokenized_document` is a list of words (not a raw string)
    return [stemmer.stem(word) for word in tokenized_document]

# again, we need to loop over each tokenized document in the list
articles_en_stemmed = [stem(doc, "english") for doc in articles_en_tokenized]
articles_de_stemmed = [stem(doc, "german") for doc in articles_de_tokenized]

# spacey equivalent:
# doc = nlp(articles_en[0])
# tokens = [token.lemma_ for token in doc]


In [ ]:
# compare original, tokenized, and stemmed versions
print("Original Article:\n", articles_en[50], "\n")
print("Tokenized Article:\n", articles_en_tokenized[50], "\n")
print("Stemmed Article:\n", articles_en_stemmed[50], "\n")

Original Article:
 If Hurricane Charley blows your house down, how can you make your insurance company pay? 

Tokenized Article:
 ['If', 'Hurricane', 'Charley', 'blows', 'your', 'house', 'down', ',', 'how', 'can', 'you', 'make', 'your', 'insurance', 'company', 'pay', '?'] 

Stemmed Article:
 ['if', 'hurrican', 'charley', 'blow', 'your', 'hous', 'down', ',', 'how', 'can', 'you', 'make', 'your', 'insur', 'compani', 'pay', '?'] 



NLTK also offers built-in stopword sets for different languages

In [ ]:
stopwords_en = set(nltkStopwords.words("english"))
stopwords_de = set(nltkStopwords.words("german"))

In [ ]:
# join the stopwords with a ", " and print them
", ".join(stopwords_en)

"aren, both, not, mightn't, ours, didn, how, mightn, haven, it's, these, hadn't, this, into, she, that, re, needn, between, do, won, yours, have, herself, in, me, we're, him, aren't, by, he, she'll, that'll, because, about, are, don't, he's, can, mustn, very, from, their, hasn't, o, y, you'd, was, same, at, as, isn't, our, then, some, does, they'll, they, yourself, has, did, out, if, wasn, too, what, her, theirs, own, needn't, its, we, doesn, it'd, few, am, no, she'd, whom, shan, couldn, for, shan't, we'd, wouldn't, them, doesn't, he'll, each, ll, than, to, there, you'll, yourselves, over, shouldn, d, didn't, who, wouldn, don, or, she's, up, doing, here, weren, myself, is, should've, off, above, i've, of, nor, which, won't, couldn't, had, where, before, once, it'll, you, hasn, hers, we've, while, hadn, s, they're, until, my, himself, again, i, during, he'd, being, a, i'm, below, under, should, so, other, ain, those, further, ve, why, i'd, more, themselves, weren't, after, down, but, wh

In [ ]:
# doing the same for the german ones
", ".join(stopwords_de)


'des, deine, dich, warst, euren, nach, da, einig, kein, hatte, jede, deinen, zu, damit, jedem, ihrem, anderes, unser, und, ob, die, ist, anderem, jenem, einen, sie, es, unter, in, jedes, um, aber, meinen, noch, dies, wieder, soll, zwar, selbst, du, war, jener, können, welchen, andere, eure, welche, zur, würde, weiter, hinter, keinem, andern, was, dort, sehr, weil, solcher, einem, wollte, jeder, welcher, vor, jenen, welchem, bist, desselben, durch, sondern, sonst, wenn, welches, hatten, während, solches, weg, einiger, doch, dieser, eures, muss, einmal, am, einigem, ihres, im, zwischen, vom, auch, seinen, von, jenes, denn, ander, ihren, nichts, als, derer, hat, seine, musste, eines, keiner, gewesen, ihre, dir, dann, wie, denselben, daß, auf, derselben, alle, etwas, ihrer, werden, wir, zum, keines, diesem, sind, keine, viel, bin, dieses, über, man, indem, euer, meinem, allen, hin, nur, habe, jeden, bis, der, könnte, ich, hab, diese, mich, dessen, machen, meiner, wirst, unsere, das, würden

We now want to remove the stopwords of our stemmed documents for further processing



In [ ]:
# Again, we need a function to apply the stopword removal to each document
def remove_stopwords(stemmed_document: list[str], stopwords: Set) -> List[str]:
    # Define a helper function that returns True if the word is NOT a stopword
    def is_stopword(word):
        return not word in stopwords

    # Filter out stopwords from the stemmed document
    # Assumes `stemmed_document` is a list of stemmed words
    return list(filter(is_stopword, stemmed_document))


articles_en_final = [remove_stopwords(doc, stopwords_en) for doc in articles_en_stemmed]
articles_de_final = [remove_stopwords(doc, stopwords_de) for doc in articles_de_stemmed]

In [ ]:
# print an article in all its versions. from the original over the tokenized, stemmed and final one
print(articles_en[50])
print(articles_en_tokenized[50])
print(articles_en_stemmed[50])
print(articles_en_final[50])

If Hurricane Charley blows your house down, how can you make your insurance company pay?
['If', 'Hurricane', 'Charley', 'blows', 'your', 'house', 'down', ',', 'how', 'can', 'you', 'make', 'your', 'insurance', 'company', 'pay', '?']
['if', 'hurrican', 'charley', 'blow', 'your', 'hous', 'down', ',', 'how', 'can', 'you', 'make', 'your', 'insur', 'compani', 'pay', '?']
['hurrican', 'charley', 'blow', 'hous', ',', 'make', 'insur', 'compani', 'pay', '?']


# Gensim

https://radimrehurek.com/gensim/

Gensim describes itself as "Topic Modelling for Humans".

We will use our NLTK-preprocessed documents as input to:

1. Build a dictionary – Mapping words to unique IDs.
2. Create a corpus – Representing documents as bag-of-words vectors.
3. Construct an index – Enabling efficient similarity queries.
4. Compute the TF-IDF matrix – Weighting terms by importance.
5. Run text queries – Searching for similar documents based on content.

This workflow demonstrates how Gensim transforms preprocessed text into powerful representations for topic modeling and similarity analysis.

In [ ]:
# Gensim is not installed by default in some environments, so we install it here
%pip install gensim


Note: you may need to restart the kernel to use updated packages.


## Building an TF-IDF model

Use a TF-IDF model to compare a user-provided input string against a trained corpus of English documents and retrieve the most similar articles.


In [ ]:
from gensim import corpora, models, similarities

# Limit the number of documents to avoid memory issues with large models
size = 500  # Adjust this value if the model is too large to run efficiently

# Create a dictionary from the first `size` documents
# The dictionary maps each word to a unique ID
corpus_dictionary_en = corpora.Dictionary(articles_en_final[:size])

# Convert each document to a Bag-of-Words (BoW) representation
# Each document becomes a list of (word_id, frequency) tuples
corpus_en = [corpus_dictionary_en.doc2bow(document) for document in articles_en_final[:size]]

# Train a TF-IDF model on the BoW corpus
# This model assigns weights to words based on their importance across documents
model_en = models.TfidfModel(corpus_en)

# Create a similarity index using the TF-IDF weighted corpus
# This allows fast similarity queries between documents
index_en = similarities.MatrixSimilarity(model_en[corpus_en])

To calculate the similarity of an input, it has to be preprocessed the same way as our training data

In [ ]:
def query_en(query_string, model_en=model_en, index_en=index_en, stopwords_en=stopwords_en, size=size, corpus_dictionary_en=corpus_dictionary_en):

    # Tokenize the query string using NLTK
    tokens = nltk.word_tokenize(query_string)

    # Apply stemming and stopword removal to the tokenized query
    # Assumes stopwords_en is a predefined set of English stopwords
    processed_query = remove_stopwords(
        stem(tokens, language="english"),
        stopwords_en
    )

    # Convert the processed query into a Bag-of-Words representation
    q = corpus_dictionary_en.doc2bow(processed_query)

    # Transform the BoW query into TF-IDF space
    q_model = model_en[q]

    # Compute similarity scores between the query and all documents
    result = index_en[q_model]

    # Sort results by similarity score in descending order
    result = sorted(enumerate(result), key=lambda item: -item[1])

    # Print the top 3 most similar documents and their scores
    for i, j in enumerate(result):
        if i > 2:
            break
        print(j, articles_en[:size][j[0]])

    # Return the full list of similarity scores with document indices
    #return result

Gensim returns the resulting document and its similarity

In [ ]:
query_en("Scientists United States", model_en, index_en);




(237, np.float32(0.3952839)) Scientists in the United States find a way to turn lazy monkeys into workaholics using gene therapy.
(450, np.float32(0.21606433)) AFP - National Basketball Association players trying to win a fourth consecutive Olympic gold medal for the United States have gotten the wake-up call that the "Dream Team" days are done even if supporters have not.
(462, np.float32(0.20889904))  ATHENS (Reuters) - The United States beat Canada in a world  best time to qualify for the final of the men's Olympic eights  race Sunday, as the two crews renewed their fierce rivalry in  front of a raucous crowd at Schinias.
